# 03 Wikidata Data Enhancement

This notebook documents the data enhancement stage of the Europeana India metadata project.

The aim of this stage is to enrich selected entities from the dataset using external authority data from Wikidata. The selected entity type is cultural heritage organisations / data providers, using the `dataProvider` column.

This follows the data enhancement task, where the goal is to identify suitable entities, check Wikidata matches, add at least one Wikidata-derived feature, and document the matching process.

## Task 1 — Peer-to-peer check-in about enhancement

### Entity type

The entity type selected for enhancement is:

**Organisations / institutions**

### Column used

The relevant column is:

`dataProvider`

This column contains the names of institutions or data providers that contributed the Europeana metadata records.

### Features that could be enhanced

Possible Wikidata-derived features include:

- Wikidata Q-ID
- standardised institution name
- country
- city or location
- coordinates
- institution type
- founding date

### Clarity of the entity type

This entity type is partly clear but may include some ambiguity.

Many providers are clearly identifiable institutions, such as:

- Royal Museums Greenwich
- Bodleian Libraries, University of Oxford
- Austrian National Library
- Deutsche Welle

However, some provider names may be ambiguous or may refer to collections rather than institutions, such as:

- Museum of Ethnography
- TopFoto
- Map Collection UK

Therefore, matching should not rely only on the name. The match should also be checked using country, description, and institution type.

## Task 2 — Manual Wikidata matching sample

For the manual Wikidata matching test, I selected a small sample of data providers from the dataset. The aim is to check whether these organisation names can be matched to Wikidata items and whether enhancement is feasible.

### Selected sample entities

| Entity from dataset | Wikidata match | Wikidata Q-ID | Feature available | Match status | Notes |
|---|---|---|---|---|---|
| Royal Museums Greenwich | Royal Museums Greenwich | Q7374509 | organisation type / location | matched | Clear match; Wikidata describes it as a network of museums in Greenwich, London. |
| Bodleian Libraries, University of Oxford | Bodleian Libraries | Q16147979 | organisation type / parent institution | matched | Clear match; Wikidata describes it as a group of research libraries at the University of Oxford. |
| Austrian National Library | Austrian National Library | Q304037 | organisation type / country | matched | Clear match; Wikidata describes it as the largest library in Austria. |
| MAK – Museum of Applied Arts | MAK – Museum of Applied Arts | Q478455 | organisation type / location | matched | Clear match; Wikidata describes it as a museum in Vienna, Austria. |

### Feasibility assessment

The manual sample suggests that Wikidata enhancement is feasible for major data providers. All four tested institutions could be matched to Wikidata items. This means that adding Wikidata Q-IDs and at least one additional feature, such as institution type or country, is promising.

### Disambiguation strategy

Some provider names may still be ambiguous. For example, “Museum of Ethnography” could refer to different museums in different countries. To disambiguate such cases, I would compare:

- the Wikidata label;
- the Wikidata description;
- country or location;
- organisation type;
- whether the item refers to a museum, library, archive, or collection.

If the match is clear, the status will be marked as `matched`. If multiple possible matches exist, the status will be marked as `uncertain`. If no suitable Wikidata item is found, the status will be marked as `not_found`.

## Task 3 — Preparing the dataset for Wikidata enhancement

The entity column selected for enhancement is `dataProvider`, because it contains the names of institutions and data providers.

For this enhancement step, I will add the following new columns:

- `wikidata_qid`: the Wikidata identifier for the matched institution;
- `wikidata_label`: the standardised institution name from Wikidata;
- `institution_type`: a simple description of the type of organisation;
- `match_status`: whether the entity was matched, uncertain, or not found.

For this first version, I will manually enrich a small sample of clearly identifiable institutions. Ambiguous institutions will be marked as `uncertain`.

In [13]:
# Import required libraries

import pandas as pd
from pathlib import Path

print("Libraries imported successfully.")

Libraries imported successfully.


In [14]:
# Define project folders safely

from pathlib import Path

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_dir = current_folder.parent
else:
    project_dir = current_folder

data_dir = project_dir / "data"
processed_dir = data_dir / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

print("Current folder:", current_folder)
print("Project folder:", project_dir)
print("Processed data folder:", processed_dir)

print("\nFiles in processed folder:")
for file in sorted(processed_dir.glob("*")):
    print("-", file.name)

Current folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\notebooks
Project folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026
Processed data folder: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed

Files in processed folder:
- europeana_india_unique_titles.csv
- europeana_india_unique_titles_enhanced_sample.csv
- europeana_india_unique_titles_enhanced_top15_providers.csv
- europeana_india_unique_titles_transformed.csv


In [15]:
# Import required libraries
# Load the transformed dataset created by notebook 02


import pandas as pd
from pathlib import Path
input_file = (
    processed_dir
    / "europeana_india_unique_titles_transformed.csv"
)

if not input_file.exists():
    raise FileNotFoundError(
        f"Transformed dataset not found: {input_file}"
    )

df = pd.read_csv(input_file)

print("Dataset loaded successfully.")
print("Input file:", input_file)
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Input file: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles_transformed.csv
Shape: (245, 12)


,query_group,search_term,title,description,subject,dataProvider,type,country,has_description,title_word_count,contains_india,possible_false_positive
0,geographic,India,India,"Barev.\n28 x 36 cm, na listu 38 x 43 cm\nMěřít...",NaN,Map Collection UK,IMAGE,['Czech Republic'],True,1,True,False
1,geographic,India,India.,"Litografie, kolor.\n28,5 x 37 cm na listu 31 x...",NaN,Map Collection UK,IMAGE,['Czech Republic'],True,1,True,False
2,geographic,India,%india%,NaN,NaN,Royal Museums Greenwich,IMAGE,['United Kingdom'],False,1,True,False
3,geographic,India,India & Nepál,"Tartalom ◊ I. ◊ Az első randevú: Delhi, 1972 ◊...",NaN,National Széchényi Library of Hungary,TEXT,['Hungary'],True,3,True,False
4,geographic,India,India SONG,NaN,NaN,National Audiovisual Institute France,SOUND,['France'],False,2,True,False


In [16]:
# Create new columns for Wikidata enhancement

df_enhanced = df.copy()

df_enhanced["wikidata_qid"] = ""
df_enhanced["wikidata_label"] = ""
df_enhanced["institution_type"] = ""
df_enhanced["match_status"] = "not_checked"

print("New enhancement columns added.")

df_enhanced[
    ["dataProvider", "wikidata_qid", "wikidata_label", "institution_type", "match_status"]
].head(10)

New enhancement columns added.


,dataProvider,wikidata_qid,wikidata_label,institution_type,match_status
0,Map Collection UK,,,,not_checked
1,Map Collection UK,,,,not_checked
2,Royal Museums Greenwich,,,,not_checked
3,National Széchényi Library of Hungary,,,,not_checked
4,National Audiovisual Institute France,,,,not_checked
5,Marucelliana Library,,,,not_checked
6,Marciana National Library,,,,not_checked
7,University Library of Genova,,,,not_checked
8,University Library of Genova,,,,not_checked
9,University Library of Genova,,,,not_checked


In [17]:
# Manual Wikidata enhancement for selected clear institution matches

manual_matches = {
    "Royal Museums Greenwich": {
        "wikidata_qid": "Q7374509",
        "wikidata_label": "Royal Museums Greenwich",
        "institution_type": "museum organisation",
        "match_status": "matched"
    },
    "Bodleian Libraries, University of Oxford": {
        "wikidata_qid": "Q16147979",
        "wikidata_label": "Bodleian Libraries",
        "institution_type": "library system",
        "match_status": "matched"
    },
    "Austrian National Library": {
        "wikidata_qid": "Q304037",
        "wikidata_label": "Austrian National Library",
        "institution_type": "national library",
        "match_status": "matched"
    },
    "MAK – Museum of Applied Arts": {
        "wikidata_qid": "Q478455",
        "wikidata_label": "MAK – Museum of Applied Arts",
        "institution_type": "museum",
        "match_status": "matched"
    },
    "Museum of Ethnography": {
        "wikidata_qid": "",
        "wikidata_label": "",
        "institution_type": "",
        "match_status": "uncertain"
    }
}

for provider, values in manual_matches.items():
    mask = df_enhanced["dataProvider"] == provider
    
    for column, value in values.items():
        df_enhanced.loc[mask, column] = value

print("Manual enhancement applied.")

df_enhanced[
    ["dataProvider", "wikidata_qid", "wikidata_label", "institution_type", "match_status"]
].drop_duplicates().head(15)

Manual enhancement applied.


,dataProvider,wikidata_qid,wikidata_label,institution_type,match_status
0,Map Collection UK,,,,not_checked
2,Royal Museums Greenwich,Q7374509,Royal Museums Greenwich,museum organisation,matched
3,National Széchényi Library of Hungary,,,,not_checked
4,National Audiovisual Institute France,,,,not_checked
5,Marucelliana Library,,,,not_checked
6,Marciana National Library,,,,not_checked
7,University Library of Genova,,,,not_checked
11,DFF – German Film Institute & Film Museum,,,,not_checked
12,Calouste Gulbenkian Foundation,,,,not_checked
13,VU University Amsterdam Library,,,,not_checked


In [18]:
# Count match status categories

match_status_counts = df_enhanced["match_status"].value_counts()

print("Match status counts:")
print(match_status_counts)

Match status counts:
match_status
not_checked    157
matched         64
uncertain       24
Name: count, dtype: int64


In [19]:
# Save enhanced sample dataset

output_file = processed_dir / "europeana_india_unique_titles_enhanced_sample.csv"

df_enhanced.to_csv(output_file, index=False, encoding="utf-8")

print("Enhanced sample dataset saved to:")
print(output_file)

Enhanced sample dataset saved to:
C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles_enhanced_sample.csv


## Interpretation of the enhancement sample

This enhancement step added Wikidata-related columns to the working dataset and manually enriched a sample of data providers.

The sample enhancement produced 64 matched rows, 24 uncertain rows, and 157 rows that were not checked yet. Clear matches were added for large and well-known institutions such as Royal Museums Greenwich, Bodleian Libraries, Austrian National Library, and MAK – Museum of Applied Arts. The provider “Museum of Ethnography” was marked as uncertain because the name is ambiguous and may refer to different institutions in different countries.

This shows that Wikidata enhancement is feasible for major institutional providers, but disambiguation is necessary for generic or ambiguous provider names.

In [20]:
# Check whether the enhanced dataset was saved

print("Enhanced dataset exists:", output_file.exists())
print("Enhanced dataset location:", output_file)

Enhanced dataset exists: True
Enhanced dataset location: C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles_enhanced_sample.csv


## Task 3 Documentation

In this data enhancement step, I enriched the `dataProvider` column, which contains the names of cultural heritage institutions and data providers.

The added enhancement columns are:

- `wikidata_qid`
- `wikidata_label`
- `institution_type`
- `match_status`

The enrichment was applied manually to a sample of clearly identifiable institutions. Clear matches were marked as `matched`, ambiguous provider names were marked as `uncertain`, and the remaining rows were left as `not_checked`.

The sample produced the following result:

- `matched`: 64 rows
- `uncertain`: 24 rows
- `not_checked`: 157 rows

For ambiguous names, such as “Museum of Ethnography,” I did not assign a Wikidata Q-ID because several institutions may have similar names. In future work, these cases should be disambiguated using additional information such as country, location, institution type, and Wikidata description.

# Week 7 Hometask — Generalising the Wikidata enhancement approach

For the hometask, I generalise the Wikidata enhancement approach from a small sample to a clearly defined larger subset of the dataset.

The selected entity type is still organisations / institutions, using the `dataProvider` column. Instead of enriching every provider in the dataset, I focus on the top 15 most frequent data providers in the transformed dataset.

This subset is useful because these providers contribute many records and are more likely to be identifiable institutions with Wikidata entries.

The enhancement adds the following columns:

- `wikidata_qid`
- `wikidata_label`
- `institution_type`
- `match_status`

The match status categories are:

- `matched`: a clear Wikidata match was identified;
- `uncertain`: the provider name is ambiguous or needs further checking;
- `not_found`: no suitable Wikidata match was identified;
- `not_checked`: the provider was outside the selected top-provider subset.

In [21]:
# Identify the top 15 data providers for hometask enhancement

top_15_providers = df_enhanced["dataProvider"].value_counts().head(15)

print("Top 15 data providers:")
print(top_15_providers)

Top 15 data providers:
dataProvider
Museum of Ethnography                                                  24
MAK – Museum of Applied Arts                                           21
Deutsche Fotothek                                                      20
Bodleian Libraries, University of Oxford                               17
Royal Museums Greenwich                                                16
German Documentation Center for Art History - Marburg Picture Index    10
Austrian National Library                                              10
International Maritime Museum                                           9
Museum of World Culture                                                 9
Cinecittà - Luce                                                        7
TopFoto                                                                 6
Toy Museum of the City of Nuremberg                                     5
Cineteca di Bologna                                                     5
Un

In [22]:
# Generalised manual Wikidata enhancement for top data providers

hometask_matches = {
    "Museum of Ethnography": {
        "wikidata_qid": "Q1371375",
        "wikidata_label": "Museum of Ethnography",
        "institution_type": "ethnographic museum",
        "match_status": "matched"
    },
    "MAK – Museum of Applied Arts": {
        "wikidata_qid": "Q478455",
        "wikidata_label": "MAK – Museum of Applied Arts",
        "institution_type": "museum",
        "match_status": "matched"
    },
    "Deutsche Fotothek": {
        "wikidata_qid": "Q655507",
        "wikidata_label": "Deutsche Fotothek",
        "institution_type": "image archive",
        "match_status": "matched"
    },
    "Bodleian Libraries, University of Oxford": {
        "wikidata_qid": "Q16147979",
        "wikidata_label": "Bodleian Libraries",
        "institution_type": "library system",
        "match_status": "matched"
    },
    "Royal Museums Greenwich": {
        "wikidata_qid": "Q7374509",
        "wikidata_label": "Royal Museums Greenwich",
        "institution_type": "museum organisation",
        "match_status": "matched"
    },
    "German Documentation Center for Art History - Marburg Picture Index": {
        "wikidata_qid": "",
        "wikidata_label": "",
        "institution_type": "image archive / documentation centre",
        "match_status": "uncertain"
    },
    "Austrian National Library": {
        "wikidata_qid": "Q304037",
        "wikidata_label": "Austrian National Library",
        "institution_type": "national library",
        "match_status": "matched"
    },
    "International Maritime Museum": {
        "wikidata_qid": "Q937608",
        "wikidata_label": "International Maritime Museum Hamburg",
        "institution_type": "maritime museum",
        "match_status": "matched"
    },
    "Museum of World Culture": {
        "wikidata_qid": "Q378377",
        "wikidata_label": "Museum of World Culture",
        "institution_type": "museum",
        "match_status": "matched"
    },
    "Cinecittà - Luce": {
        "wikidata_qid": "",
        "wikidata_label": "",
        "institution_type": "film archive / media institution",
        "match_status": "uncertain"
    },
    "TopFoto": {
        "wikidata_qid": "",
        "wikidata_label": "",
        "institution_type": "image archive",
        "match_status": "uncertain"
    },
    "Toy Museum of the City of Nuremberg": {
        "wikidata_qid": "",
        "wikidata_label": "",
        "institution_type": "museum",
        "match_status": "uncertain"
    },
    "Cineteca di Bologna": {
        "wikidata_qid": "Q1092493",
        "wikidata_label": "Cineteca di Bologna",
        "institution_type": "film archive",
        "match_status": "matched"
    },
    "University Library of Genova": {
        "wikidata_qid": "Q3639620",
        "wikidata_label": "University Library of Genoa",
        "institution_type": "university library",
        "match_status": "matched"
    },
    "Deutsche Welle": {
        "wikidata_qid": "Q153770",
        "wikidata_label": "Deutsche Welle",
        "institution_type": "public broadcaster",
        "match_status": "matched"
    }
}

# Start from the existing enhanced dataset
df_hometask_enhanced = df_enhanced.copy()

# Mark all rows as not_checked first
df_hometask_enhanced["match_status"] = "not_checked"

# Apply manual matches to the selected top providers
for provider, values in hometask_matches.items():
    mask = df_hometask_enhanced["dataProvider"] == provider
    
    for column, value in values.items():
        df_hometask_enhanced.loc[mask, column] = value

print("Hometask enhancement applied to top 15 providers.")

df_hometask_enhanced[
    ["dataProvider", "wikidata_qid", "wikidata_label", "institution_type", "match_status"]
].drop_duplicates().head(20)

Hometask enhancement applied to top 15 providers.


,dataProvider,wikidata_qid,wikidata_label,institution_type,match_status
0,Map Collection UK,,,,not_checked
2,Royal Museums Greenwich,Q7374509,Royal Museums Greenwich,museum organisation,matched
3,National Széchényi Library of Hungary,,,,not_checked
4,National Audiovisual Institute France,,,,not_checked
5,Marucelliana Library,,,,not_checked
6,Marciana National Library,,,,not_checked
7,University Library of Genova,Q3639620,University Library of Genoa,university library,matched
11,DFF – German Film Institute & Film Museum,,,,not_checked
12,Calouste Gulbenkian Foundation,,,,not_checked
13,VU University Amsterdam Library,,,,not_checked


In [23]:
# Count match status after hometask enhancement

hometask_status_counts = df_hometask_enhanced["match_status"].value_counts()

print("Hometask match status counts:")
print(hometask_status_counts)

Hometask match status counts:
match_status
matched        139
not_checked     78
uncertain       28
Name: count, dtype: int64


In [24]:
# Save the final top-15 provider enrichment dataset

final_output_file = (
    processed_dir
    / "europeana_india_unique_titles_enhanced_top15_providers.csv"
)

df_hometask_enhanced.to_csv(
    final_output_file,
    index=False,
    encoding="utf-8"
)

print("Final enhanced dataset saved to:")
print(final_output_file)
print("File exists:", final_output_file.exists())
print("Final shape:", df_hometask_enhanced.shape)

Final enhanced dataset saved to:
C:\Users\kevin\Documents\GitHub\Open_Project_2026\data\processed\europeana_india_unique_titles_enhanced_top15_providers.csv
File exists: True
Final shape: (245, 16)


## Interpretation of the hometask enhancement

For the hometask, I generalised the Wikidata enhancement approach to the top 15 most frequent data providers in the transformed dataset.

After improving the matching, the result shows that 139 rows were matched, 28 rows were marked as uncertain, and 78 rows were not checked because they were outside the selected top-15 provider subset.

This suggests that Wikidata enhancement is feasible for many major institutional providers in the dataset. Well-known organisations such as Royal Museums Greenwich, Bodleian Libraries, Austrian National Library, Deutsche Welle, Deutsche Fotothek, Museum of Ethnography, University Library of Genova, and MAK – Museum of Applied Arts could be matched with Wikidata-style identifiers and institution types.

Two previously uncertain providers were upgraded to matched after checking Wikidata more carefully: Museum of Ethnography and University Library of Genova.

However, some providers remain uncertain because their names are ambiguous, generic, or may refer to collections rather than formal institutions. These cases require further disambiguation using additional information such as country, location, Wikidata description, and organisation type.

This enhancement improves the dataset by making institutional metadata more structured and easier to compare.